## Cell 1 - Title / Notes

In [1]:
# ============================================================
# 01_build_dataset.ipynb
# Hybrid dataset builder:
#   AoA/phase  -> geometry branch
#   amplitude  -> fingerprint / confidence / correction branch
# ============================================================
#
# Output:
#   source_labeled.npz
#   source_unlabeled.npz
#   target_support.npz
#   target_query.npz
#   label_map.json
#   dataset_summary.json
#
# Assumption:
#   MATLAB already exported per-view files separately:
#       A/amp_window_xxxxx.npy
#       A/ampavg_window_xxxxx.npy   (optional)
#       A/pha_window_xxxxx.npy
#       A/meta_xxxxx.mat
#       B/...
# ============================================================

## Cell 2 - Import

In [2]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat

## Cell 3A - Config

In [3]:
# ============================================================
# CONFIG
# ============================================================

from pathlib import Path
import random
import numpy as np

DATA_ROOT = Path("/home/tonyliao/Location_AMP")   # change this
OUT_DIR   = DATA_ROOT / "dataset_build_hybrid"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATA_ROOT / "Train"
PRED_DIR  = DATA_ROOT / "Pred"

AMP_PATTERN    = "amp_window_*.npy"
AMPAVG_PATTERN = "ampavg_window_*.npy"
PHA_PATTERN    = "pha_window_*.npy"
META_PATTERN   = "meta_*.mat"

USE_RAW_AMP = True
USE_AVECSI  = True
USE_PHASE   = True

KEEP_FLAGGED_WINDOWS = True
MIN_WIN_SCORE_FILTER = None

TRAIN_UNLABELED_RATIO   = 0.30
PRED_SUPPORT_PER_CLASS  = 20

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## Cell 3B - Label

In [4]:
EMPTY_CLASS_NAMES = {"empty", "empty_pred", "none", "vacant", "no_person"}

LABEL_NORMALIZATION = {
    "Empty": "Empty",
    # "Empty_Pred": "Empty",

    "LeftDown": "LeftDown",
    # "LeftDown_Far": "LeftDown",

    "LeftMid": "LeftMid",

    "LeftUp": "LeftUp",
    # "LeftUp_Near": "LeftUp",
    # "LeftUp_Pred": "LeftUp",

    "MiddleDown": "MiddleDown",
    "MiddleUp": "MiddleUp",

    "RightDown": "RightDown",
    "RightMid": "RightMid",
    "RightUp": "RightUp",
}

## Cell 4 - Helper

In [5]:
def natural_key(s): ## for natural sorting
    s = str(s)
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

def sorted_glob(folder: Path, pattern: str): ## glob with natural sorting
    return sorted(folder.glob(pattern), key=lambda p: natural_key(p.name))

def infer_label_from_folder(folder_name: str): ## infer label name from folder name, with normalization
    folder_name = folder_name.strip()
    return LABEL_NORMALIZATION.get(folder_name, folder_name)

def is_empty_label(label_name: str): ## check if a label name is considered empty
    return label_name.lower() in EMPTY_CLASS_NAMES

def build_label_map_from_names(label_names): # build label map from a list of label names, sorted naturally
    uniq = sorted(set(label_names))
    return {name: idx for idx, name in enumerate(uniq)}
    uniq = sorted(set(label_names), key=natural_key)
    return {name: idx for idx, name in enumerate(uniq)}

def safe_scalar(x, default=np.nan): ## safely convert to scalar float, return default if fails
    try:
        arr = np.asarray(x).squeeze()
        if arr.size == 0:
            return default
        return float(arr.flat[0])
    except Exception:
        return default

def safe_vector(x): ## safely convert to 1D float32 vector, return empty array if fails
    try:
        arr = np.asarray(x).squeeze()
        if arr.size == 0:
            return np.array([], dtype=np.float32)
        return arr.astype(np.float32).reshape(-1)
    except Exception:
        return np.array([], dtype=np.float32)


def load_meta_mat(meta_path: Path): ## load meta .mat file and extract relevant fields, return a dict with default NaN values
    out = {
        "amp_mean": np.nan,
        "amp_std": np.nan,
        "amp_var": np.nan,
        "amp_range": np.nan,
        "amp_energy": np.nan,
        "ampn_std": np.nan,
        "phase_std": np.nan,
        "ampavg_mean": np.nan,
        "ampavg_std": np.nan,
        "ampavg_energy": np.nan,
        "win_score": np.nan,
        "amp_score": np.nan,
        "pha_score": np.nan,
        "corr_score": np.nan,
        "is_flagged_bad": np.nan,
        "z_sub_amp_std": np.array([], dtype=np.float32),
        "z_sub_ampn_std": np.array([], dtype=np.float32),
        "z_sub_energy": np.array([], dtype=np.float32),
    }

    try:
        mat = loadmat(meta_path, squeeze_me=True, struct_as_record=False)
        meta = mat.get("meta", None)
        if meta is None:
            return out

        scalar_keys = [
            "amp_mean", "amp_std", "amp_var", "amp_range", "amp_energy",
            "ampn_std", "phase_std", "ampavg_mean", "ampavg_std",
            "ampavg_energy", "win_score", "amp_score", "pha_score",
            "corr_score", "is_flagged_bad"
        ]
        vector_keys = ["z_sub_amp_std", "z_sub_ampn_std", "z_sub_energy"]

        for k in scalar_keys:
            if hasattr(meta, k):
                out[k] = safe_scalar(getattr(meta, k))

        for k in vector_keys:
            if hasattr(meta, k):
                out[k] = safe_vector(getattr(meta, k))

    except Exception as e:
        print(f"[WARN] failed to read meta {meta_path}: {e}")

    return out

def load_one_view_folder(view_dir: Path): # load one view folder and return a list of dicts with keys: id, amp, ampavg, pha, meta
    amp_files    = sorted_glob(view_dir, AMP_PATTERN)
    ampavg_files = sorted_glob(view_dir, AMPAVG_PATTERN)
    pha_files    = sorted_glob(view_dir, PHA_PATTERN)
    meta_files   = sorted_glob(view_dir, META_PATTERN)

    amp_map    = {f.stem.replace("amp_window_", ""): f for f in amp_files}
    ampavg_map = {f.stem.replace("ampavg_window_", ""): f for f in ampavg_files}
    pha_map    = {f.stem.replace("pha_window_", ""): f for f in pha_files}
    meta_map   = {f.stem.replace("meta_", ""): f for f in meta_files}

    ids = set(meta_map.keys())
    if USE_RAW_AMP:
        ids &= set(amp_map.keys())
    if USE_PHASE:
        ids &= set(pha_map.keys())

    ids = sorted(ids, key=natural_key)

    rows = []
    for wid in ids:
        rows.append({
            "id": wid,
            "amp": amp_map.get(wid, None),
            "ampavg": ampavg_map.get(wid, None),
            "pha": pha_map.get(wid, None),
            "meta": meta_map.get(wid, None),
        })
    return rows

def pair_two_views(loc_dir: Path): # pair two views in the same location folder, return a list of dicts with keys: id, A_amp, A_ampavg, A_pha, A_meta, B_amp, B_ampavg, B_pha, B_meta
    a_dir = loc_dir / "A"
    b_dir = loc_dir / "B"

    if not a_dir.exists() or not b_dir.exists():
        return []

    A = load_one_view_folder(a_dir)
    B = load_one_view_folder(b_dir)

    A_map = {x["id"]: x for x in A}
    B_map = {x["id"]: x for x in B}

    common_ids = sorted(set(A_map.keys()) & set(B_map.keys()), key=natural_key)

    pairs = []
    for wid in common_ids:
        a = A_map[wid]
        b = B_map[wid]

        if USE_AVECSI:
            if a["ampavg"] is None or b["ampavg"] is None:
                continue

        pairs.append({
            "id": wid,
            "A_amp": a["amp"],
            "A_ampavg": a["ampavg"],
            "A_pha": a["pha"],
            "A_meta": a["meta"],
            "B_amp": b["amp"],
            "B_ampavg": b["ampavg"],
            "B_pha": b["pha"],
            "B_meta": b["meta"],
        })

    return pairs

def scan_domain(domain_dir: Path, domain_name: str):
    rows = []

    if not domain_dir.exists():
        print(f"[WARN] missing folder: {domain_dir}")
        return pd.DataFrame()

    class_dirs = sorted([p for p in domain_dir.iterdir() if p.is_dir()], key=lambda p: p.name.lower())

    for class_dir in class_dirs:
        raw_folder_name = class_dir.name
        label_name = infer_label_from_folder(raw_folder_name)
        presence = 0 if is_empty_label(label_name) else 1

        pairs = pair_two_views(class_dir)

        for item in pairs:
            rows.append({
                "domain": domain_name,               # "train" or "pred"
                "raw_folder_name": raw_folder_name,  # original folder name
                "label_name": label_name,            # normalized label
                "presence": presence,
                "pair_id": item["id"],
                "loc_dir": str(class_dir),

                "A_amp": str(item["A_amp"]) if item["A_amp"] else "",
                "A_ampavg": str(item["A_ampavg"]) if item["A_ampavg"] else "",
                "A_pha": str(item["A_pha"]) if item["A_pha"] else "",
                "A_meta": str(item["A_meta"]) if item["A_meta"] else "",

                "B_amp": str(item["B_amp"]) if item["B_amp"] else "",
                "B_ampavg": str(item["B_ampavg"]) if item["B_ampavg"] else "",
                "B_pha": str(item["B_pha"]) if item["B_pha"] else "",
                "B_meta": str(item["B_meta"]) if item["B_meta"] else "",
            })

    return pd.DataFrame(rows)

## Cell 5 - Scan src and target

In [6]:
df_train = scan_domain(TRAIN_DIR, "train")
df_pred  = scan_domain(PRED_DIR, "pred")

print("df_train:", df_train.shape)
print("df_pred :", df_pred.shape)

display(df_train.head())
display(df_pred.head())

df_train: (2535, 14)
df_pred : (1010, 14)


,domain,raw_folder_name,label_name,presence,pair_id,loc_dir,A_amp,A_ampavg,A_pha,A_meta,B_amp,B_ampavg,B_pha,B_meta
0,train,Empty,Empty,0,00001,/home/tonyliao/Location_AMP/Train/Empty,/home/tonyliao/Location_AMP/Train/Empty/A/amp_...,/home/tonyliao/Location_AMP/Train/Empty/A/ampa...,/home/tonyliao/Location_AMP/Train/Empty/A/pha_...,/home/tonyliao/Location_AMP/Train/Empty/A/meta...,/home/tonyliao/Location_AMP/Train/Empty/B/amp_...,/home/tonyliao/Location_AMP/Train/Empty/B/ampa...,/home/tonyliao/Location_AMP/Train/Empty/B/pha_...,/home/tonyliao/Location_AMP/Train/Empty/B/meta...
1,train,Empty,Empty,0,00002,/home/tonyliao/Location_AMP/Train/Empty,/home/tonyliao/Location_AMP/Train/Empty/A/amp_...,/home/tonyliao/Location_AMP/Train/Empty/A/ampa...,/home/tonyliao/Location_AMP/Train/Empty/A/pha_...,/home/tonyliao/Location_AMP/Train/Empty/A/meta...,/home/tonyliao/Location_AMP/Train/Empty/B/amp_...,/home/tonyliao/Location_AMP/Train/Empty/B/ampa...,/home/tonyliao/Location_AMP/Train/Empty/B/pha_...,/home/tonyliao/Location_AMP/Train/Empty/B/meta...
2,train,Empty,Empty,0,00003,/home/tonyliao/Location_AMP/Train/Empty,/home/tonyliao/Location_AMP/Train/Empty/A/amp_...,/home/tonyliao/Location_AMP/Train/Empty/A/ampa...,/home/tonyliao/Location_AMP/Train/Empty/A/pha_...,/home/tonyliao/Location_AMP/Train/Empty/A/meta...,/home/tonyliao/Location_AMP/Train/Empty/B/amp_...,/home/tonyliao/Location_AMP/Train/Empty/B/ampa...,/home/tonyliao/Location_AMP/Train/Empty/B/pha_...,/home/tonyliao/Location_AMP/Train/Empty/B/meta...
3,train,Empty,Empty,0,00004,/home/tonyliao/Location_AMP/Train/Empty,/home/tonyliao/Location_AMP/Train/Empty/A/amp_...,/home/tonyliao/Location_AMP/Train/Empty/A/ampa...,/home/tonyliao/Location_AMP/Train/Empty/A/pha_...,/home/tonyliao/Location_AMP/Train/Empty/A/meta...,/home/tonyliao/Location_AMP/Train/Empty/B/amp_...,/home/tonyliao/Location_AMP/Train/Empty/B/ampa...,/home/tonyliao/Location_AMP/Train/Empty/B/pha_...,/home/tonyliao/Location_AMP/Train/Empty/B/meta...
4,train,Empty,Empty,0,00005,/home/tonyliao/Location_AMP/Train/Empty,/home/tonyliao/Location_AMP/Train/Empty/A/amp_...,/home/tonyliao/Location_AMP/Train/Empty/A/ampa...,/home/tonyliao/Location_AMP/Train/Empty/A/pha_...,/home/tonyliao/Location_AMP/Train/Empty/A/meta...,/home/tonyliao/Location_AMP/Train/Empty/B/amp_...,/home/tonyliao/Location_AMP/Train/Empty/B/ampa...,/home/tonyliao/Location_AMP/Train/Empty/B/pha_...,/home/tonyliao/Location_AMP/Train/Empty/B/meta...


,domain,raw_folder_name,label_name,presence,pair_id,loc_dir,A_amp,A_ampavg,A_pha,A_meta,B_amp,B_ampavg,B_pha,B_meta
0,pred,Center,Center,1,00001,/home/tonyliao/Location_AMP/Pred/Center,/home/tonyliao/Location_AMP/Pred/Center/A/amp_...,/home/tonyliao/Location_AMP/Pred/Center/A/ampa...,/home/tonyliao/Location_AMP/Pred/Center/A/pha_...,/home/tonyliao/Location_AMP/Pred/Center/A/meta...,/home/tonyliao/Location_AMP/Pred/Center/B/amp_...,/home/tonyliao/Location_AMP/Pred/Center/B/ampa...,/home/tonyliao/Location_AMP/Pred/Center/B/pha_...,/home/tonyliao/Location_AMP/Pred/Center/B/meta...
1,pred,Center,Center,1,00002,/home/tonyliao/Location_AMP/Pred/Center,/home/tonyliao/Location_AMP/Pred/Center/A/amp_...,/home/tonyliao/Location_AMP/Pred/Center/A/ampa...,/home/tonyliao/Location_AMP/Pred/Center/A/pha_...,/home/tonyliao/Location_AMP/Pred/Center/A/meta...,/home/tonyliao/Location_AMP/Pred/Center/B/amp_...,/home/tonyliao/Location_AMP/Pred/Center/B/ampa...,/home/tonyliao/Location_AMP/Pred/Center/B/pha_...,/home/tonyliao/Location_AMP/Pred/Center/B/meta...
2,pred,Center,Center,1,00003,/home/tonyliao/Location_AMP/Pred/Center,/home/tonyliao/Location_AMP/Pred/Center/A/amp_...,/home/tonyliao/Location_AMP/Pred/Center/A/ampa...,/home/tonyliao/Location_AMP/Pred/Center/A/pha_...,/home/tonyliao/Location_AMP/Pred/Center/A/meta...,/home/tonyliao/Location_AMP/Pred/Center/B/amp_...,/home/tonyliao/Location_AMP/Pred/Center/B/ampa...,/home/tonyliao/Location_AMP/Pred/Center/B/pha_...,/home/tonyliao/Location_AMP/Pred/Center/B/meta...
3,pred,Center,Center,1,00004,/home/tonyliao/Location_AMP/Pred/Center,/home/tonyliao/Location_AMP/Pred/Center/A/amp_...,/home/tonyliao/Location_AMP/Pred/Center/A/ampa...,/home/tonyliao/Location_AMP/Pred/Center/A/pha_...,/home/tonyliao/Location_AMP/Pred/Center/A/meta...,/home/tonyliao/Location_AMP/Pred/Center/B/amp_...,/home/tonyliao/Location_AMP/Pred/Center/B/ampa...,/home/tonyliao/Location_AMP/Pred/Center/B/pha_...,/home/tonyliao/Location_AMP/Pred/Center/B/meta...
4,pred,Center,Center,1,00005,/home/tonyliao/Location_AMP/Pred/Center,/home/tonyliao/Location_AMP/Pred/Center/A/amp_...,/home/tonyliao/Location_AMP/Pred/Center/A/ampa...,/home/tonyliao/Location_AMP/Pred/Center/A/pha_...,/home/tonyliao/Location_AMP/Pred/Center/A/meta...,/home/tonyliao/Location_AMP/Pred/Center/B/amp_...,/home/tonyliao/Location_AMP/Pred/Center/B/ampa...,/home/tonyliao/Location_AMP/Pred/Center/B/pha_...,/home/tonyliao/Location_AMP/Pred/Center/B/meta...


## Cell 6 - Basic Check

In [7]:
assert len(df_train) > 0, "No paired train samples found."
assert len(df_pred) > 0, "No paired pred samples found."

print("\nTrain raw folder counts:")
print(df_train["raw_folder_name"].value_counts().sort_index())

print("\nTrain normalized label counts:")
print(df_train["label_name"].value_counts().sort_index())

print("\nPred raw folder counts:")
print(df_pred["raw_folder_name"].value_counts().sort_index())

print("\nPred normalized label counts:")
print(df_pred["label_name"].value_counts().sort_index())


Train raw folder counts:
raw_folder_name
Empty         297
LeftDown      296
LeftMid       273
LeftUp        292
MiddleDown    268
MiddleUp      295
RightDown     232
RightMid      289
RightUp       293
Name: count, dtype: int64

Train normalized label counts:
label_name
Empty         297
LeftDown      296
LeftMid       273
LeftUp        292
MiddleDown    268
MiddleUp      295
RightDown     232
RightMid      289
RightUp       293
Name: count, dtype: int64

Pred raw folder counts:
raw_folder_name
Center          147
Corner          136
Empty_Pred      147
LeftDown_Far    147
LeftUp_Near     145
LeftUp_Pred     147
Wall            141
Name: count, dtype: int64

Pred normalized label counts:
label_name
Center          147
Corner          136
Empty_Pred      147
LeftDown_Far    147
LeftUp_Near     145
LeftUp_Pred     147
Wall            141
Name: count, dtype: int64


## Cell 7 - Build Shared Label Map

In [8]:
all_label_names = sorted(set(df_train["label_name"]) | set(df_pred["label_name"]))
label_map = build_label_map_from_names(all_label_names)
inv_label_map = {v: k for k, v in label_map.items()}

df_train["label_id"] = df_train["label_name"].map(label_map)
df_pred["label_id"]  = df_pred["label_name"].map(label_map)

print(label_map)

{'Center': 0, 'Corner': 1, 'Empty': 2, 'Empty_Pred': 3, 'LeftDown': 4, 'LeftDown_Far': 5, 'LeftMid': 6, 'LeftUp': 7, 'LeftUp_Near': 8, 'LeftUp_Pred': 9, 'MiddleDown': 10, 'MiddleUp': 11, 'RightDown': 12, 'RightMid': 13, 'RightUp': 14, 'Wall': 15}


## Cell 8 — optional inspection of one sample

In [9]:
sample_row = df_train.iloc[0]

if USE_RAW_AMP:
    A_amp = np.load(sample_row["A_amp"])
    B_amp = np.load(sample_row["B_amp"])
    print("A_amp:", A_amp.shape, A_amp.dtype)
    print("B_amp:", B_amp.shape, B_amp.dtype)

if USE_AVECSI:
    A_ampavg = np.load(sample_row["A_ampavg"])
    B_ampavg = np.load(sample_row["B_ampavg"])
    print("A_ampavg:", A_ampavg.shape, A_ampavg.dtype)
    print("B_ampavg:", B_ampavg.shape, B_ampavg.dtype)

if USE_PHASE:
    A_pha = np.load(sample_row["A_pha"])
    B_pha = np.load(sample_row["B_pha"])
    print("A_pha:", A_pha.shape, A_pha.dtype)
    print("B_pha:", B_pha.shape, B_pha.dtype)

A_amp: (256, 41, 2) float32
B_amp: (256, 41, 2) float32
A_ampavg: (1, 41, 2) float32
B_ampavg: (1, 41, 2) float32
A_pha: (256, 41, 2) float32
B_pha: (256, 41, 2) float32


## Cell 9 — load metadata into dataframe columns

In [10]:
def attach_meta_columns(df: pd.DataFrame):
    df = df.copy()

    scalar_cols = [
        "amp_mean", "amp_std", "amp_var", "amp_range", "amp_energy",
        "ampn_std", "phase_std", "ampavg_mean", "ampavg_std",
        "ampavg_energy", "win_score", "amp_score", "pha_score",
        "corr_score", "is_flagged_bad"
    ]

    for prefix in ["A", "B"]:
        for c in scalar_cols:
            df[f"{prefix}_{c}"] = np.nan

    A_sub_amp_std, A_sub_ampn_std, A_sub_energy = [], [], []
    B_sub_amp_std, B_sub_ampn_std, B_sub_energy = [], [], []

    for idx, row in df.iterrows():
        metaA = load_meta_mat(Path(row["A_meta"])) if row["A_meta"] else {}
        metaB = load_meta_mat(Path(row["B_meta"])) if row["B_meta"] else {}

        for c in scalar_cols:
            df.at[idx, f"A_{c}"] = metaA.get(c, np.nan)
            df.at[idx, f"B_{c}"] = metaB.get(c, np.nan)

        A_sub_amp_std.append(metaA.get("z_sub_amp_std", np.array([], dtype=np.float32)))
        A_sub_ampn_std.append(metaA.get("z_sub_ampn_std", np.array([], dtype=np.float32)))
        A_sub_energy.append(metaA.get("z_sub_energy", np.array([], dtype=np.float32)))

        B_sub_amp_std.append(metaB.get("z_sub_amp_std", np.array([], dtype=np.float32)))
        B_sub_ampn_std.append(metaB.get("z_sub_ampn_std", np.array([], dtype=np.float32)))
        B_sub_energy.append(metaB.get("z_sub_energy", np.array([], dtype=np.float32)))

    df["A_z_sub_amp_std"] = A_sub_amp_std
    df["A_z_sub_ampn_std"] = A_sub_ampn_std
    df["A_z_sub_energy"] = A_sub_energy

    df["B_z_sub_amp_std"] = B_sub_amp_std
    df["B_z_sub_ampn_std"] = B_sub_ampn_std
    df["B_z_sub_energy"] = B_sub_energy

    return df

## Cell 10 — apply meta attachment

In [11]:
df_train = attach_meta_columns(df_train)
df_pred = attach_meta_columns(df_pred)

display(df_train.head(3))

KeyboardInterrupt: 

## Cell 11 — quality filter

In [ ]:
def quality_filter(df: pd.DataFrame, keep_flagged=True, min_win_score=None):
    df = df.copy()
    mask = np.ones(len(df), dtype=bool)

    if not keep_flagged:
        mask &= (df["A_is_flagged_bad"].fillna(0).values == 0)
        mask &= (df["B_is_flagged_bad"].fillna(0).values == 0)

    if min_win_score is not None:
        mask &= (df["A_win_score"].fillna(-np.inf).values >= min_win_score)
        mask &= (df["B_win_score"].fillna(-np.inf).values >= min_win_score)

    return df.loc[mask].reset_index(drop=True)


KEEP_FLAGGED_WINDOWS = True
MIN_WIN_SCORE_FILTER = None

df_train = quality_filter(
    df_train,
    keep_flagged=KEEP_FLAGGED_WINDOWS,
    min_win_score=MIN_WIN_SCORE_FILTER
)

df_pred = quality_filter(
    df_pred,
    keep_flagged=KEEP_FLAGGED_WINDOWS,
    min_win_score=MIN_WIN_SCORE_FILTER
)

print("After filter:")
print("train:", df_train.shape)
print("pred :", df_pred.shape)

After filter:
train: (2535, 51)
pred : (1010, 51)


## Cell 12 — split source into labeled / unlabeled

In [ ]:
def split_train_labeled_unlabeled(df_train, unlabeled_ratio=0.30, seed=42):
    labeled_parts = []
    unlabeled_parts = []

    for label_id, g in df_train.groupby("label_id"):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n_unlab = int(round(len(g) * unlabeled_ratio))
        n_unlab = min(max(n_unlab, 0), len(g))

        unlabeled_parts.append(g.iloc[:n_unlab].copy())
        labeled_parts.append(g.iloc[n_unlab:].copy())

    df_lab = pd.concat(labeled_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df_unlab = pd.concat(unlabeled_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return df_lab, df_unlab


df_train_labeled, df_train_unlabeled = split_train_labeled_unlabeled(
    df_train,
    unlabeled_ratio=TRAIN_UNLABELED_RATIO,
    seed=SEED,
)

print("train_labeled:", df_train_labeled.shape)
print("train_unlabeled:", df_train_unlabeled.shape)

train_labeled: (1774, 51)
train_unlabeled: (761, 51)


## Cell 13 — split target into support / query

In [ ]:
def split_pred_support_query(df_pred, support_per_class=20, seed=42):
    support_parts = []
    query_parts = []

    for label_id, g in df_pred.groupby("label_id"):
        g = g.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n_support = min(support_per_class, len(g))

        support_parts.append(g.iloc[:n_support].copy())
        query_parts.append(g.iloc[n_support:].copy())

    df_support = pd.concat(support_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df_query = pd.concat(query_parts, axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

    return df_support, df_query


df_pred_support, df_pred_query = split_pred_support_query(
    df_pred,
    support_per_class=PRED_SUPPORT_PER_CLASS,
    seed=SEED,
)

print("pred_support:", df_pred_support.shape)
print("pred_query:", df_pred_query.shape)

pred_support: (140, 51)
pred_query: (870, 51)


## Cell 14 — summary tables

In [ ]:
def show_split_summary(name, df):
    print(f"\n{name}: {df.shape}")
    print("raw folder counts:")
    print(df["raw_folder_name"].value_counts().sort_index())
    print("\nnormalized label counts:")
    print(df["label_name"].value_counts().sort_index())


show_split_summary("TRAIN LABELED", df_train_labeled)
show_split_summary("TRAIN UNLABELED", df_train_unlabeled)
show_split_summary("PRED SUPPORT", df_pred_support)
show_split_summary("PRED QUERY", df_pred_query)


TRAIN LABELED: (1774, 51)
raw folder counts:
raw_folder_name
Empty         208
LeftDown      207
LeftMid       191
LeftUp        204
MiddleDown    188
MiddleUp      207
RightDown     162
RightMid      202
RightUp       205
Name: count, dtype: int64

normalized label counts:
label_name
Empty         208
LeftDown      207
LeftMid       191
LeftUp        204
MiddleDown    188
MiddleUp      207
RightDown     162
RightMid      202
RightUp       205
Name: count, dtype: int64

TRAIN UNLABELED: (761, 51)
raw folder counts:
raw_folder_name
Empty         89
LeftDown      89
LeftMid       82
LeftUp        88
MiddleDown    80
MiddleUp      88
RightDown     70
RightMid      87
RightUp       88
Name: count, dtype: int64

normalized label counts:
label_name
Empty         89
LeftDown      89
LeftMid       82
LeftUp        88
MiddleDown    80
MiddleUp      88
RightDown     70
RightMid      87
RightUp       88
Name: count, dtype: int64

PRED SUPPORT: (140, 51)
raw folder counts:
raw_folder_name
Center 

## Cell 15 — feature packing helper

In [ ]:
def pack_one_dataframe(df: pd.DataFrame):
    obj = {
        "A_amp_paths": df["A_amp"].astype(str).values,
        "B_amp_paths": df["B_amp"].astype(str).values,

        "A_ampavg_paths": df["A_ampavg"].astype(str).values,
        "B_ampavg_paths": df["B_ampavg"].astype(str).values,

        "A_pha_paths": df["A_pha"].astype(str).values,
        "B_pha_paths": df["B_pha"].astype(str).values,

        "A_meta_paths": df["A_meta"].astype(str).values,
        "B_meta_paths": df["B_meta"].astype(str).values,

        "label_id": df["label_id"].astype(np.int64).values,
        "label_name": df["label_name"].astype(str).values,
        "presence": df["presence"].astype(np.int64).values,
        "domain": df["domain"].astype(str).values,
        "pair_id": df["pair_id"].astype(str).values,
        "loc_dir": df["loc_dir"].astype(str).values,
    }

    scalar_cols = [
        "amp_mean", "amp_std", "amp_var", "amp_range", "amp_energy",
        "ampn_std", "phase_std", "ampavg_mean", "ampavg_std",
        "ampavg_energy", "win_score", "amp_score", "pha_score",
        "corr_score", "is_flagged_bad"
    ]

    for prefix in ["A", "B"]:
        for c in scalar_cols:
            obj[f"{prefix}_{c}"] = df[f"{prefix}_{c}"].astype(np.float32).values

    obj["A_z_sub_amp_std"]  = np.array(df["A_z_sub_amp_std"].tolist(), dtype=object)
    obj["A_z_sub_ampn_std"] = np.array(df["A_z_sub_ampn_std"].tolist(), dtype=object)
    obj["A_z_sub_energy"]   = np.array(df["A_z_sub_energy"].tolist(), dtype=object)

    obj["B_z_sub_amp_std"]  = np.array(df["B_z_sub_amp_std"].tolist(), dtype=object)
    obj["B_z_sub_ampn_std"] = np.array(df["B_z_sub_ampn_std"].tolist(), dtype=object)
    obj["B_z_sub_energy"]   = np.array(df["B_z_sub_energy"].tolist(), dtype=object)

    return obj

## Cell 16 — save datasets

In [ ]:
train_labeled_obj   = pack_one_dataframe(df_train_labeled)
train_unlabeled_obj = pack_one_dataframe(df_train_unlabeled)
pred_support_obj    = pack_one_dataframe(df_pred_support)
pred_query_obj      = pack_one_dataframe(df_pred_query)

np.savez_compressed(OUT_DIR / "train_labeled.npz", **train_labeled_obj)
np.savez_compressed(OUT_DIR / "train_unlabeled.npz", **train_unlabeled_obj)
np.savez_compressed(OUT_DIR / "pred_support.npz", **pred_support_obj)
np.savez_compressed(OUT_DIR / "pred_query.npz", **pred_query_obj)

with open(OUT_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "label_map": label_map,
            "inv_label_map": {str(k): v for k, v in inv_label_map.items()},
            "label_normalization": LABEL_NORMALIZATION,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

summary = {
    "data_root": str(DATA_ROOT),
    "train_dir": str(TRAIN_DIR),
    "pred_dir": str(PRED_DIR),
    "use_raw_amp": USE_RAW_AMP,
    "use_avecsi": USE_AVECSI,
    "use_phase": USE_PHASE,
    "train_labeled_n": int(len(df_train_labeled)),
    "train_unlabeled_n": int(len(df_train_unlabeled)),
    "pred_support_n": int(len(df_pred_support)),
    "pred_query_n": int(len(df_pred_query)),
    "num_classes": int(len(label_map)),
    "labels": label_map,
    "seed": SEED,
}

with open(OUT_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Saved to:", OUT_DIR)

Saved to: /home/tonyliao/Location_AMP/dataset_build_hybrid


## Cell 17 — quick reload test

In [ ]:
tmp = np.load(OUT_DIR / "train_labeled.npz", allow_pickle=True)

print("keys:", tmp.files)
print("num samples:", len(tmp["label_id"]))
print("first A_amp path:", tmp["A_amp_paths"][0])
print("first B_amp path:", tmp["B_amp_paths"][0])
print("first A_pha path:", tmp["A_pha_paths"][0])
print("first B_pha path:", tmp["B_pha_paths"][0])
print("first label:", tmp["label_id"][0], tmp["label_name"][0])

keys: ['A_amp_paths', 'B_amp_paths', 'A_ampavg_paths', 'B_ampavg_paths', 'A_pha_paths', 'B_pha_paths', 'A_meta_paths', 'B_meta_paths', 'label_id', 'label_name', 'presence', 'domain', 'pair_id', 'loc_dir', 'A_amp_mean', 'A_amp_std', 'A_amp_var', 'A_amp_range', 'A_amp_energy', 'A_ampn_std', 'A_phase_std', 'A_ampavg_mean', 'A_ampavg_std', 'A_ampavg_energy', 'A_win_score', 'A_amp_score', 'A_pha_score', 'A_corr_score', 'A_is_flagged_bad', 'B_amp_mean', 'B_amp_std', 'B_amp_var', 'B_amp_range', 'B_amp_energy', 'B_ampn_std', 'B_phase_std', 'B_ampavg_mean', 'B_ampavg_std', 'B_ampavg_energy', 'B_win_score', 'B_amp_score', 'B_pha_score', 'B_corr_score', 'B_is_flagged_bad', 'A_z_sub_amp_std', 'A_z_sub_ampn_std', 'A_z_sub_energy', 'B_z_sub_amp_std', 'B_z_sub_ampn_std', 'B_z_sub_energy']
num samples: 1774
first A_amp path: /home/tonyliao/Location_AMP/Train/MiddleUp/A/amp_window_00070.npy
first B_amp path: /home/tonyliao/Location_AMP/Train/MiddleUp/B/amp_window_00070.npy
first A_pha path: /home/tonyl

## Cell 18 — optional loader preview for later notebooks

In [ ]:
def load_pair_arrays_from_npz_row(npz_obj, idx):
    out = {
        "label_id": int(npz_obj["label_id"][idx]),
        "label_name": str(npz_obj["label_name"][idx]),
        "presence": int(npz_obj["presence"][idx]),
    }

    if USE_RAW_AMP:
        out["A_amp"] = np.load(str(npz_obj["A_amp_paths"][idx]))
        out["B_amp"] = np.load(str(npz_obj["B_amp_paths"][idx]))

    if USE_AVECSI:
        out["A_ampavg"] = np.load(str(npz_obj["A_ampavg_paths"][idx]))
        out["B_ampavg"] = np.load(str(npz_obj["B_ampavg_paths"][idx]))

    if USE_PHASE:
        out["A_pha"] = np.load(str(npz_obj["A_pha_paths"][idx]))
        out["B_pha"] = np.load(str(npz_obj["B_pha_paths"][idx]))

    return out


example = load_pair_arrays_from_npz_row(tmp, 0)
for k, v in example.items():
    if isinstance(v, np.ndarray):
        print(k, v.shape, v.dtype)
    else:
        print(k, v)

label_id 11
label_name MiddleUp
presence 1
A_amp (256, 41, 2) float32
B_amp (256, 41, 2) float32
A_ampavg (1, 41, 2) float32
B_ampavg (1, 41, 2) float32
A_pha (256, 41, 2) float32
B_pha (256, 41, 2) float32
